In [0]:
from pyspark.sql.functions import col, coalesce, lit, date_format

# Aliases
df_op = spark.read.table("ecommerce_analytics.silver.order_products").alias("op")
df_so = spark.read.table("ecommerce_analytics.silver.sales_orders").alias("so")
df_pr = spark.read.table("ecommerce_analytics.silver.promotions").alias("pr")

# Join (INCLUDING correct promotion join)
df_fact = (
    df_op
    .join(
        df_so,
        col("op.order_number") == col("so.order_number"),
        "left"
    )
    .join(
        df_pr,
        (col("op.order_number") == col("pr.order_number")) &
        (col("op.id") == col("pr.promo_product_id")),
        "left"
    )
)

# Transform
df_fact = df_fact.select(
    col("op.order_number"),
    col("so.customer_id"),
    col("op.id").alias("product_id"),

    # Date FK
    date_format(col("so.order_timestamp"), "yyyyMMdd").cast("int").alias("order_date_key"),

    # Measures
    col("op.qty").cast("int").alias("qty"),
    col("op.price").cast("double").alias("unit_price"),
    coalesce(col("pr.discount"), lit(0)).alias("discount"),
    col("pr.promo_quantity").cast("int").alias("promo_quantity")
)

# Derived column
df_fact = df_fact.withColumn(
    "total_amount",
    col("qty") * col("unit_price") * (1 - col("discount"))
)

In [0]:
# Save correctly
df_fact.write.mode("overwrite").saveAsTable("ecommerce_analytics.gold.fact_sales")

In [0]:
df_fact.display()